<a href="https://colab.research.google.com/github/gianrodriguezd/ai_engineering_exercises/blob/main/Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Optimization:
Test Function: Rosenbrock function

The purpose of this exercise is not only to calcualte the loss function but to use three optimization methods to find the lowest point in the loss landscape and calcualate its loss so one can use the output to see which one had a lower loss

In [14]:
def rosenbrock(params):
    x, y = params
    return (1 - x) ** 2 + 100 * (y - x ** 2) ** 2

##Definition of partial derivatives
def rosenbrock_gradient(params):
    x, y = params
    df_dx = -2 * (1 - x) + 200 * (y - x ** 2) * (-2 * x)
    df_dy = 200 * (y - x ** 2)
    return [df_dx, df_dy]

##definition of the gradient descent
class GradientDescent:
    def __init__(self, lr=0.001): ##initialization of class and def of learning rate
        self.lr = lr
 ##returns new values of x and y taking into account
      ## their grads and lr defined above
    def step(self, params, grads):
        return [p - self.lr * g for p, g in zip(params, grads)]

class SGDMomentum:
  ##Momentum class for SDG is defined and initialized
  ## lr, momentum and velocity are initialized
    def __init__(self, lr=0.001, momentum=0.9):
        self.lr = lr
        self.momentum = momentum
        self.velocity = None

#steps are defined considering momentum theory
#velocity is initially set as 0, only at the very beginning
## velocity then calculated for all grads
    def step(self, params, grads):
        if self.velocity is None:
            self.velocity = [0.0] * len(params) ##sets velocity as 0\

            ##takes v=0 and starts calculating velocity for all grads
        self.velocity = [
            self.momentum * v + g
            for v, g in zip(self.velocity, grads)
        ]

        ##returns parameters - lr * velocity for all
        ##params and velocities zipped.
        return [p - self.lr * v for p, v in zip(params, self.velocity)]

      ##adam class initialization
class Adam:
    def __init__(self, lr=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8):

      ##initialization of m,v and t. lr, beta1,2 and epsilon values are saved.
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.m = None
        self.v = None
        self.t = 0


    def step(self, params, grads):
        if self.m is None:

          ##m and v are set to 0
            self.m = [0.0] * len(params)
            self.v = [0.0] * len(params)

        ##count of t increasing by one
        self.t += 1

        self.m = [
            self.beta1 * m + (1 - self.beta1) * g
            for m, g in zip(self.m, grads)
        ] ## m = beta1*m + (1-beta1)*gradient
        self.v = [
            self.beta2 * v + (1 - self.beta2) * g ** 2
            for v, g in zip(self.v, grads)
        ] ## v = beta2*v + (1-beta2)* (gradient^2)

        ##definition of mhat and vhat.
        m_hat = [m / (1 - self.beta1 ** self.t) for m in self.m] ## m/1-(b1^t)
        v_hat = [v / (1 - self.beta2 ** self.t) for v in self.v] ## v/1-(b2^t)

        return [
            p - self.lr * mh / (vh ** 0.5 + self.epsilon)
            for p, mh, vh in zip(params, m_hat, v_hat)
        ] ##returns new position after applying adam, integrating vh, mh and epsilon


#defines an optimizer to run, 5000 steps.
def optimize(optimizer, func, grad_func, start, steps=5000):
  ##optimizer param is where we'll call either class previously defined
  ##func: loss function passed to the function
  ##grad_func: func that calculates gradients (eg: grad rosenrbrock)
  ##start: initial positions of x and y
    params = list(start) ##uses list to make a copy of start and uses this as initial x and y
    history = [params[:]] ## as code runs creates a copy with all entries of params
    for _ in range(steps):

      ## uses params to calculate grads
        grads = grad_func(params)
        ##uses the class through optimizer and calculates step using params and grads
        params = optimizer.step(params, grads)
        history.append(params[:]) ##appends position to history list
    return history ##returns history

start = [-1.0, 1.0] ##defines start positions


##testing all classes/optimization methods
gd_history = optimize(GradientDescent(lr=0.0005), rosenbrock, rosenbrock_gradient, start)
sgd_history = optimize(SGDMomentum(lr=0.0001, momentum=0.9), rosenbrock, rosenbrock_gradient, start)
adam_history = optimize(Adam(lr=0.01), rosenbrock, rosenbrock_gradient, start)

print("")
for name, history in [("GD", gd_history), ("SGD+M", sgd_history), ("Adam", adam_history)]:
    final = history[-1] ##outputs the last position in the history list
    loss = rosenbrock(final) ##uses the last position on the rosenbrock function for the loss
    print(f"{name:6s} -> x={final[0]:.6f}, y={final[1]:.6f}, loss={loss:.8f}")


GD     -> x=0.798131, y=0.636104, loss=0.04083385
SGD+M  -> x=0.940412, y=0.884127, loss=0.00355685
Adam   -> x=0.999992, y=0.999984, loss=0.00000000


Through the results the adaptative learning rate method (ADAM) had the best results with a loss of 0.00 almost reaching the expected global minimum of the rosenbrock function.